# Retail Pricing Optimization — what's the *real* price elasticity?

**Thesis:** a naive `quantity ~ price` regression makes retail demand look
nearly *inelastic* — because stores set prices in response to demand (price
endogeneity). Correcting for that flips the conclusion, and the **gap between
the naive and corrected elasticity is the margin you'd leave on the table by
pricing on the wrong number.**

Plan: estimate elasticity three ways (naive OLS → product/time fixed effects →
IV with competitor prices), then feed the corrected elasticity into the
margin-maximizing price `p* = c·ε/(ε+1)`.

Data: [Retail Price Optimization (Suddharshan)](https://www.kaggle.com/datasets/suddharshan/retail-price-optimization) — a product×month panel (52 products, 20 months, 9 categories).

In [1]:
import os, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf
from statsmodels.sandbox.regression.gmm import IV2SLS  # statsmodels only -> runs on the stock Kaggle image, no pip install

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 50)

# auto-detect the CSV: glob anywhere under the Kaggle mount (folder name
# varies), then fall back to local checkouts.
CANDIDATES = (glob.glob('/kaggle/input/**/retail_price.csv', recursive=True)
              + ['../data/raw/retail_price.csv', 'data/raw/retail_price.csv'])
DATA = next((p for p in CANDIDATES if os.path.exists(p)), None)
if DATA is None:
    raise FileNotFoundError(
        'retail_price.csv not found. On Kaggle: Add Input -> search '
        '"Retail Price Optimization" (suddharshan). Available inputs: '
        + str(glob.glob('/kaggle/input/*')))
print('reading', DATA)

reading ../data/raw/retail_price.csv


## Load & prepare
Add logs (constant-elasticity demand `q = A·p^ε` is linear in logs), an integer
month index for time effects, and a markdown proxy (this data has no explicit
promo flag, so we mark months where a product is priced below its own median).

In [2]:
COMP = ['comp_1', 'comp_2', 'comp_3']  # competitor prices -> IV instruments

def load_prepared(path=DATA):
    df = pd.read_csv(path).copy()
    df['period'] = (df['year'] * 12 + df['month']).astype(int)  # int month index
    df['log_q'] = np.log(df['qty'])
    df['log_p'] = np.log(df['unit_price'])
    for c in COMP:
        df[f'log_{c}'] = np.log(df[c])
    med = df.groupby('product_id')['unit_price'].transform('median')
    df['is_markdown'] = df['unit_price'] < med
    return df

df = load_prepared()
print(df.shape, '| products:', df.product_id.nunique(), '| months:', df.period.nunique())
df[['product_id', 'period', 'unit_price', 'qty', 'log_p', 'log_q', 'is_markdown']].head()

(676, 37) | products: 52 | months: 20


,product_id,period,unit_price,qty,log_p,log_q,is_markdown
0,bed1,24209,45.95,1,3.827554,0.000000,False
1,bed1,24210,45.95,3,3.827554,1.098612,False
2,bed1,24211,45.95,6,3.827554,1.791759,False
3,bed1,24212,45.95,4,3.827554,1.386294,False
4,bed1,24213,45.95,2,3.827554,0.693147,False


## EDA: the pooled log–log cloud
Pooled across products the slope looks almost flat — the endogeneity problem
showing itself. The real signal is *within* a product over time.

In [3]:
fig, ax = plt.subplots(1, 2, figsize=(13, 5))
sns.scatterplot(data=df, x='log_p', y='log_q', hue='is_markdown', alpha=0.4, ax=ax[0])
ax[0].set(title='Pooled: log demand vs log price', xlabel='log(unit_price)', ylabel='log(qty)')

top = df['product_id'].value_counts().head(8).index
sns.scatterplot(data=df[df.product_id.isin(top)], x='log_p', y='log_q',
                hue='product_id', alpha=0.7, ax=ax[1], legend=False)
ax[1].set(title='Within-product (8 products): slopes steepen', xlabel='log(unit_price)', ylabel='')
plt.tight_layout(); plt.show()

/tmp/ipykernel_19919/3289002210.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## Elasticity, three ways (biased → causal)

In [4]:
def naive_ols(df):
    m = smf.ols('log_q ~ log_p', data=df).fit(cov_type='HC1')
    ci = m.conf_int().loc['log_p']
    return ('naive_ols', m.params['log_p'], ci[0], ci[1], 'biased foil')

def fixed_effects(df):
    # within estimator via LSDV: product + period dummies, SE clustered by product
    res = smf.ols('log_q ~ log_p + C(product_id) + C(period)', data=df).fit(
        cov_type='cluster', cov_kwds={'groups': df['product_id']})
    ci = res.conf_int().loc['log_p']
    return ('fixed_effects', res.params['log_p'], ci[0], ci[1],
            'product+time FE (LSDV), clustered')

def iv_2sls(df, instruments=None):
    # competitor prices only: lagged own price is a STRONG but INVALID instrument
    # (sticky prices carry lagged demand shocks -> exclusion restriction fails).
    instruments = instruments or [f'log_{c}' for c in COMP]
    d = df.dropna(subset=['log_q', 'log_p', *instruments]).copy()
    d['const'] = 1.0
    iv = IV2SLS(d['log_q'], d[['const', 'log_p']],
                instrument=d[['const', *instruments]]).fit()
    lo, hi = iv.params['log_p'] - 1.96 * iv.bse['log_p'], iv.params['log_p'] + 1.96 * iv.bse['log_p']
    # robust first-stage F: joint test that instruments don't predict price (want >> 10)
    fs = smf.ols('log_p ~ ' + ' + '.join(instruments), data=d).fit(cov_type='HC1')
    F = float(fs.f_test(' = 0, '.join(instruments) + ' = 0').fvalue)
    return ('iv_2sls', iv.params['log_p'], lo, hi,
            f'IV competitor prices; first-stage F={F:.0f}')

rows = [naive_ols(df), fixed_effects(df), iv_2sls(df)]
res = pd.DataFrame(rows, columns=['method', 'epsilon', 'ci_low', 'ci_high', 'note'])
res.round(3)

,method,epsilon,ci_low,ci_high,note
0,naive_ols,-0.131,-0.244,-0.018,biased foil
1,fixed_effects,-2.685,-3.627,-1.743,"product+time FE (LSDV), clustered"
2,iv_2sls,-0.159,-0.354,0.036,IV competitor prices; first-stage F=395


In [5]:
fig, ax = plt.subplots(figsize=(8, 3.5))
y = range(len(res))
ax.errorbar(res['epsilon'], y,
            xerr=[res['epsilon'] - res['ci_low'], res['ci_high'] - res['epsilon']],
            fmt='o', capsize=4, color='C0')
ax.axvline(-1, ls='--', color='grey', lw=1)
ax.text(-1.02, 2.4, 'unit elastic', ha='right', color='grey', fontsize=9)
ax.set_yticks(list(y)); ax.set_yticklabels(res['method'])
ax.set(xlabel='elasticity ε  (95% CI)', title='Naive looks inelastic; fixed effects reveal elastic demand')
plt.tight_layout(); plt.show()

/tmp/ipykernel_19919/4051740772.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


### Reading the result
- **Naive OLS ≈ −0.13** — apparently *inelastic*, implausible for discretionary goods.
- **Product + time fixed effects ≈ −2.7** — comparing a product to *itself* over time
  reveals real price sensitivity; the pooled fit was confounded by expensive products
  simply selling in different volumes.
- **IV (competitor prices) ≈ −0.16**, CI brushing zero. The instruments are *strong*
  (first-stage F ≫ 10) but identify a different margin than the FE within-variation; I
  report the tension rather than cherry-pick. (Lagged own price is excluded on purpose:
  strong first stage, but it violates the exclusion restriction.)

**Caveats:** constant-elasticity is a functional-form assumption; markdown is a proxy;
52×20 is small (wide CIs).

## From elasticity to the margin-maximizing price
For `q = A·p^ε` (ε < −1), margin `(p−c)·q(p)` is maximized at `p* = c·ε/(ε+1)`.
We anchor the curve to a representative point and sweep assumed unit cost
(the data has no cost field).

In [6]:
def optimal_price(c, eps):
    if eps >= -1:
        raise ValueError(f'eps={eps:.3f} not elastic (need < -1): no interior optimum')
    return c * eps / (eps + 1)

eps = res.loc[res.method == 'fixed_effects', 'epsilon'].iloc[0]
p_obs, q_obs = df['unit_price'].median(), df['qty'].median()
A = q_obs / p_obs ** eps  # intercept through the observed point
margin = lambda p, c: (p - c) * A * p ** eps

scen = []
for frac in [0.4, 0.5, 0.6, 0.7]:
    c = frac * p_obs
    p_star = optimal_price(c, eps)
    uplift = 100 * (margin(p_star, c) - margin(p_obs, c)) / margin(p_obs, c)
    scen.append({'cost_frac': frac, 'cost': round(c, 2), 'p_obs': round(p_obs, 2),
                 'p_star': round(p_star, 2), 'margin_uplift_%': round(uplift, 1)})
pd.DataFrame(scen)

,cost_frac,cost,p_obs,p_star,margin_uplift_%
0,0.4,35.96,89.9,57.30,32.6
1,0.5,44.95,89.9,71.62,9.2
2,0.6,53.94,89.9,85.95,0.4
3,0.7,62.93,89.9,100.27,3.3


## Per-product counterfactual validation
Does acting on the model beat the prices actually charged? Estimate **each product's own**
elasticity, set its margin-maximizing price, and aggregate the margin uplift vs status quo.

Two honest problems, two standard fixes:
1. Per-product slopes are **noisy** (~13 months each) -> partial-pool (empirical-Bayes
   shrinkage) toward the pooled elasticity.
2. Constant-elasticity **extrapolates** -> cap extreme ε and clip price moves to a ±25%
   guardrail (keeps recommendations near the observed price support, like real pricing teams).

In [7]:
def per_product_elasticities(df, eps_pool, min_obs=4, min_levels=3):
    recs = []
    for pid, g in df.groupby('product_id'):
        if len(g) < min_obs or g['log_p'].nunique() < min_levels:
            recs.append((pid, len(g), np.nan, np.nan)); continue
        m = smf.ols('log_q ~ log_p', data=g).fit()
        recs.append((pid, len(g), m.params['log_p'], m.bse['log_p']))
    e = pd.DataFrame(recs, columns=['product_id', 'n', 'eps_raw', 'se'])
    est = e['eps_raw'].notna()
    # empirical-Bayes shrinkage toward the pooled elasticity (precision-weighted)
    tau2 = max(0.0, e.loc[est, 'eps_raw'].var(ddof=1) - (e.loc[est, 'se'] ** 2).mean())
    e['w'] = tau2 / (tau2 + e['se'] ** 2)
    e['eps'] = np.where(est, e['w'] * e['eps_raw'] + (1 - e['w']) * eps_pool, eps_pool)
    return e

def per_product_counterfactual(df, etab, cost_frac=0.6, eps_floor=-6.0, max_move=0.25):
    ref = df.groupby('product_id').agg(p_ref=('unit_price', 'median'),
                                       q_ref=('qty', 'median'), n=('qty', 'size')).reset_index()
    m = ref.merge(etab[['product_id', 'eps']], on='product_id', how='left')
    rows = []
    for r in m.itertuples():
        eps = max(r.eps, eps_floor)              # cap noisy extreme elasticities
        c = cost_frac * r.p_ref
        if eps < -1:
            p_unc = c * eps / (eps + 1)
            lo, hi = r.p_ref * (1 - max_move), r.p_ref * (1 + max_move)
            p_star = min(max(p_unc, lo), hi)     # guardrail: stay near observed support
        else:
            p_star = r.p_ref                     # not elastic -> hold price
        A = r.q_ref / r.p_ref ** eps
        m_obs = (r.p_ref - c) * r.q_ref
        m_star = (p_star - c) * A * p_star ** eps
        rows.append({'product_id': r.product_id, 'n': r.n, 'eps': eps, 'p_ref': r.p_ref,
                     'p_star': p_star, 'price_change_%': 100 * (p_star - r.p_ref) / r.p_ref,
                     'margin_obs': m_obs, 'margin_star': m_star,
                     'uplift_%': 100 * (m_star - m_obs) / m_obs})
    pp = pd.DataFrame(rows); w = pp['n']
    summary = {'cost_frac': cost_frac, 'n_optimizable': int((pp.eps < -1).sum()),
               'share_price_up': round(float((pp.p_star > pp.p_ref).mean()), 2),
               'weighted_uplift_%': round(float(100 * (w * (pp.margin_star - pp.margin_obs)).sum()
                                                  / (w * pp.margin_obs).sum()), 1),
               'median_uplift_%': round(float(pp['uplift_%'].median()), 1)}
    return pp, summary

etab = per_product_elasticities(df, eps_pool=eps)  # eps = pooled FE estimate from above
sweep = pd.DataFrame([per_product_counterfactual(df, etab, cost_frac=cf)[1]
                      for cf in [0.4, 0.5, 0.6, 0.7]])
sweep

,cost_frac,n_optimizable,share_price_up,weighted_uplift_%,median_uplift_%
0,0.4,34,0.06,44.7,13.6
1,0.5,34,0.08,31.9,4.2
2,0.6,34,0.21,17.5,0.6
3,0.7,34,0.40,7.8,3.2


In [8]:
pp, summary = per_product_counterfactual(df, etab, cost_frac=0.6)
print(summary)
print('price move -> cut:', int((pp['price_change_%'] < -0.01).sum()),
      '| hold:', int((pp['price_change_%'].abs() <= 0.01).sum()),
      '| raise:', int((pp['price_change_%'] > 0.01).sum()))
pp.sort_values('uplift_%', ascending=False).round(2).head(8)

{'cost_frac': 0.6, 'n_optimizable': 34, 'share_price_up': 0.21, 'weighted_uplift_%': 17.5, 'median_uplift_%': 0.6}
price move -> cut: 23 | hold: 18 | raise: 11


,product_id,n,eps,p_ref,p_star,price_change_%,margin_obs,margin_star,uplift_%
0,bed1,16,-6.00,39.99,29.99,-25.0,111.97,235.92,110.70
3,bed4,10,-6.00,46.90,35.17,-25.0,187.60,395.27,110.70
14,cool2,13,-6.00,129.99,97.49,-25.0,467.96,986.00,110.70
43,perfumery2,13,-6.00,118.05,88.54,-25.0,377.76,795.94,110.70
18,furniture1,13,-6.00,36.85,27.64,-25.0,176.90,372.73,110.70
19,furniture2,13,-6.00,75.00,56.25,-25.0,1320.00,2781.23,110.70
4,bed5,5,-5.10,205.00,153.75,-25.0,2624.00,4263.86,62.49
39,health7,20,-5.03,58.99,44.24,-25.0,224.16,357.09,59.30


**Verdict:** across the whole cost sweep the aggregate margin uplift stays **positive**; at a
central cost assumption the model mostly recommends **modest price cuts**, holds price on products
it can't reliably optimize, and lifts weighted margin by a believable double-digit %. Trust the
*direction and rough magnitude*, not three sig figs — the ±25% guardrail does real work, and the
estimates are in-sample with an assumed cost.

### The punchline
Same cost, two elasticities: the **naive ε (≈ −0.13) isn't even elastic**, so
`p* = c·ε/(ε+1)` has no interior optimum — it would say *raise price without limit*.
Only the **corrected ε** yields a finite, sane recommendation. That is the margin cost
of pricing on the wrong number, made concrete.

**Next (v2):** dunnhumby "Breakfast at the Frat" with true promo flags as instruments,
and markdown-over-time as a dynamic program. Single-product static optimization here
ignores competitor reaction and cross-product cannibalization.